In [10]:
import os  
import base64
from tqdm import tqdm
import time
from openai import AzureOpenAI  
from dotenv import load_dotenv
load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT_URL_2", "")  
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "")  
subscription_key = os.getenv("AZURE_OPENAI_API_KEY_2", "")  

# Initialize Azure OpenAI Service client with key-based authentication    
client = AzureOpenAI(  
    azure_endpoint=endpoint,  
    api_key=subscription_key,  
    api_version="2024-05-01-preview",
)


In [8]:
import pandas as pd
validation_data= pd.read_csv("../type_classification-validation.csv")

In [9]:
#Prepare the chat prompt 
new_validation_df = pd.DataFrame(columns=["Sentence", "Result"])

print("Running labelling")
for index, row in tqdm(validation_data.iterrows(), total=len(validation_data)):
    chat_prompt = [{
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": "You are an AI Model that help to classify whether a sentence can be used to support information to create Class Diagram or Use Case Diagram or Activity Diagram\n\nBelow are given 5 sentences and whether it is useful for any of the diagram\n1. \"AP : As a case progresses , I need to record all the individuals and organizations that Verdict: take part in the case activities and the specific role they play .\"\nUseful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n2. \"Unless you are a celebrity or a good friend of Romano you will need a reservation .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n3. \"Therefore , there can be overlapping table reservations .\"\nVerdict: Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n4. \"These samples are sometimes sub - divided and distributed to multiple research teams or labs for different specialized observations .\"\nVerdict: Not Useful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n5. \"When the reservation party arrives at Romano 's the reservation is assigned to one waiter .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n6. \"Yes , we do have customers rent two or more vehicles at the same time .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n7. \"Geological samples are retrieved from the field and then processed in the laboratory to determine various properties , including chemistry , mineralogy , age , and petrophysical properties like density , porosity , permeability .\"\nVerdict: Useful for Class Diagram, Not Useful for Use Case Diagram, Useful for Activity Diagram\n\n8. \"For a hygienist 's appointment , preparation could be as simple as seating the patient in dental chair and putting a bib around his or her neck .\"\nVerdict: Not Useful for Class Diagram, Not Useful for Use Case Diagram, Useful for Activity Diagram\n\n9. \"AP : Actually , that is a constant source of confusion and pain .\"\nVerdict: Not Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n10. \"Romano tends to overbook tables .\"\nVerdict: Not Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n11. \"Checks are received by mail .\nVerdict: Not Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n12. \"We have 347 rental offices across the western United States .\"\nVerdict: Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n13. \"There are several restaurant managers who report to Romano .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n14. \"If a customer damaged a vehicle , abandoned it , or did n’t fully pay the bill , then we tag the customer as a poor risk , and wo n’t rent to that customer again .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n15. \"The clinic maintains a supplies inventory file that a worker fills out once a week by physically inspecting each of the three procedures rooms .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n\nUser will put a sentence and decide whether it will be useful for any of the category as an example output like this: [Useful Class, Useful Use Case, Not Useful Activity]\n\n\n\n"
            }
        ]
    }, {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": row['sentence']
            }
        ]
    }]
    
    if index % 30 == 0 and index != 0:
        time.sleep(60)
        
    # Generate the completion  
    completion = client.chat.completions.create(  
        model=deployment,
        messages=chat_prompt,
        max_tokens=800,  
        temperature=0.7,  
        top_p=0.95,  
        frequency_penalty=0,  
        presence_penalty=0,
        stop=None,  
        stream=False
    )
    result = completion.choices[0].message.content
    new_validation_df = new_validation_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
    
new_validation_df.to_csv("gpt4o-label-validation-15 example.csv")
    

Running labelling


  0%|          | 0/145 [00:00<?, ?it/s]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_21908\89755035.py:40: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_validation_df = new_validation_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
  1%|          | 1/145 [00:01<04:02,  1.68s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_21908\89755035.py:40: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_validation_df = new_validation_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
  1%|▏         | 2/145 [00:02<02:38,  1.11s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_21908\89755035.py:40: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_validation_df = new_validati